In [ ]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
def clean_id(json_text):
    list_names = []
    converted_list = ast.literal_eval(json_text)
    for item in converted_list:
        list_names.append(item["name"])
    return list_names

def cleaning_and_preprocessing(df):
    df.dropna(inplace=True)
    df["release_date"] = pd.to_datetime(df["release_date"], errors='coerce')
    df["release_date"] = df["release_date"].dt.strftime('%d/%m/%Y')
    df.rename(columns={"title_x": "title"}, inplace=True)
    df["genres"] = df["genres"].apply(clean_id)
    df["keywords"] = df["keywords"].apply(clean_id)
    df["cast"] = df["cast"].apply(clean_id)
    df["crew"] = df["crew"].apply(clean_id)
    df["production_companies"] = df["production_companies"].apply(clean_id)
    df["production_countries"] = df["production_countries"].apply(clean_id)
    df["spoken_languages"] = df["spoken_languages"].apply(clean_id)
    df.drop(["id", "movie_id", "title_y", "budget", "revenue", "production_countries"], axis=1, inplace=True)
    return df
    
#def normalize_columns(df, columns):
    for col in columns:
        df[col] = df[col].apply(lambda x: [str(elemento).lower().replace(" ", "") for elemento in x] if isinstance(x, list) else x)
    return df 

In [4]:
movies = pd.read_csv("Dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("Dataset/tmdb_5000_credits.csv")
df = pd.merge(movies, credits, left_on="id", right_on="movie_id")

df = cleaning_and_preprocessing(df)   
#df = normalize_columns(df, ["genres", "production_companies", "crew", "spoken_languages", "cast"])
df.to_csv('tmdb_5000_pulito.csv', index=False)
print("Dataset salvato con successo!")

Dataset salvato con successo!


In [ ]:
### Sentence Embedding
df["cast"] = df["cast"].apply(lambda x: " ".join(x))
df["crew"] = df["crew"].apply(lambda x: " ".join(x))
df["keywords"] = df["keywords"].apply(lambda x: " ".join(x))
df["production_companies"] = df["production_companies"].apply(lambda x: " ".join(x))
sentence_var = (df["overview"] + " " + df["cast"] + " " + df["crew"] + " " + df["keywords"] + " " + df["tagline"] +" " + df["production_companies"]) 
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_sentence = model.encode(sentence_var.tolist())

### Cosine Similarity